# WiseNet — Simulation 48 Heures Rigoureuse et Reproductible

**Auteurs** : WiseNet Research Team  
**Dépôt** : [samya818/spatial-son-milp](https://github.com/samya818/spatial-son-milp)  
**Dataset** : Telecom Italia Big Data Challenge — Milan, novembre 2013  
**Licence dataset** : Creative Commons Attribution 4.0  

---

## Ce que ce notebook démontre

Ce notebook permet à toute personne — y compris un jury — de **reproduire exactement** les résultats
de la simulation 48 heures de WiseNet et de comprendre la logique de comparaison.

### Les trois politiques comparées

| | Politique | Description | Déployable en prod ? |
|---|---|---|---|
| **A** | **Statique** | Aucun ajustement. Chaque cellule garde son offset à 0 dB. | ✅ Oui (c'est l'état actuel des réseaux sans SON) |
| **B** | **WiseNet ML+MILP** | XGBoost prédit le trafic t+1, MILP optimise, décision appliquée sur V_réel | ✅ Oui (notre contribution) |
| **C** | **Oracle MILP** | MILP optimise en connaissant parfaitement le futur | ❌ Non (impossible en vrai) |

> **Pourquoi comparer avec l'Oracle ?** Pour mesurer le *regret d'incertitude* : quelle fraction du gain
> théorique maximum notre modèle XGBoost capte-t-il réellement ?

### Fenêtre temporelle — Justification du choix des 2 jours

Nous avons sélectionné **2013-11-12 + 2013-11-13** car c'est la paire de jours consécutifs
avec le volume total maximal du dataset (88.35 To sur 48h).  
Cela maximise la pression sur le réseau et rend les gains du MILP les plus significatifs.

## 0. Vérification de l'environnement et reproductibilité

Avant toute chose, on vérifie que les fichiers source n'ont pas changé (SHA-256).

In [ ]:
import hashlib, json, platform
from pathlib import Path

ROOT = Path("..")

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(65536), b''):
            h.update(chunk)
    return h.hexdigest()

FEATURES_PATH = ROOT / 'research/data/processed/features_target_1024cells.parquet'
MODEL_PATH    = ROOT / 'research/models/xgb_q80.pkl'
META_PATH     = ROOT / 'research/reports/sim48h/sim48h_metadata.json'

sha_data  = sha256_file(FEATURES_PATH)
sha_model = sha256_file(MODEL_PATH)

print(f'Dataset SHA-256 : {sha_data}')
print(f'Modele  SHA-256 : {sha_model}')

if META_PATH.exists():
    meta = json.loads(META_PATH.read_text())
    ok_d = sha_data  == meta['dataset_sha256']
    ok_m = sha_model == meta['model_sha256']
    print(f'\nDataset identique aux resultats publies : {"✅ OUI" if ok_d else "❌ NON — fichier modifie !"}')
    print(f'Modele  identique aux resultats publies : {"✅ OUI" if ok_m else "❌ NON — fichier modifie !"}')
else:
    print('\n⚠️  Metadonnees non trouvees. Lancez d abord scripts/simulation_48h_rigorous.py')

## 1. Chargement des résultats de la simulation

In [ ]:
import polars as pl
import numpy as np
import json

SLOTS_CSV    = ROOT / 'research/reports/sim48h/sim48h_slots.csv'
SUMMARY_JSON = ROOT / 'research/reports/sim48h/sim48h_summary.json'

df = pl.read_csv(SLOTS_CSV)
summary = json.loads(SUMMARY_JSON.read_text())

print(f'Slots charges : {len(df)}')
print(f'Colonnes      : {df.columns}')
df.head(5)

## 2. Métriques agrégées — Tableau scientifique complet

Ce tableau reproduit le bilan de la simulation script.

In [ ]:
g = summary['gains']
v = summary['volumes_total_mo']
r = summary['reliability']
p = summary['performance']

print('=' * 65)
print('  BILAN SCIENTIFIQUE — 48 HEURES (96 SLOTS DE 30 MIN)')
print('=' * 65)
print(f"  Demande reelle totale        : {v['real_demand']/1e6:.2f} To")
print(f"  Slots ameliores (B < A)      : {r['slots_improved']} / {r['slots_total']} ({r['slots_improved_pct']}%)")
print('-' * 65)
print(f"  {'Politique':<28} {'Insatisfait (Mo)':>16} {'Insatisfait (Go)':>16} {'Gain':>8}")
print('-' * 65)
print(f"  {'A — Statique (ref.)':<28} {v['static_unsatisfied']:>16,.1f} {v['static_unsatisfied']/1024:>16,.2f} {'—':>8}")
print(f"  {'B — WiseNet ML+MILP':<28} {v['milp_unsatisfied']:>16,.1f} {v['milp_unsatisfied']/1024:>16,.2f} {g['milp_vs_static_pct']:>7.3f}%")
print(f"  {'C — Oracle MILP (theorique)':<28} {v['oracle_unsatisfied']:>16,.1f} {v['oracle_unsatisfied']/1024:>16,.2f} {g['oracle_vs_static_pct']:>7.3f}%")
print('=' * 65)
print(f"\n  Efficacite ML    : {g['ml_efficiency_pct']:.2f}% du gain Oracle capture")
print(f"  Regret total     : {g['regret_ml_go']:.3f} Go / 48h")
print(f"  Gain absolu      : {g['milp_vs_static_go']:.3f} Go en moins de congestion")
print(f"  Temps MILP moy.  : {p['avg_milp_solve_s']:.2f}s / slot")
print(f"  MAE XGBoost moy. : {p['avg_mae_mo_per_cell']:.3f} Mo/maille/slot")

## 3. Vérification de la logique de comparaison — Points critiques

### 3.1 Absence de fuite de données (data leakage)

La décision SON au slot t est calculée sur **V̂(t+1)** (prédit), JAMAIS sur V_réel(t+1).  
V_réel(t+1) n'est utilisé qu'en lecture seule pour **évaluer** la congestion résiduelle après la décision.

```
Instant t              Instant t+1
──────────────────     ────────────────────────────────
Features(t) ──►  XGBoost ──► V̂(t+1)
                                │
                              MILP ──► z*(t)    ← décision basée sur prédiction
                                                    ↓
                             V_réel(t+1) ──► évaluation(z*(t))  ← ground truth
```

### 3.2 Conservation de masse

Le trafic délésté d'une cellule surchargée est **reçu** par les cellules voisines.  
La somme totale du trafic est constante à chaque slot.

In [ ]:
# Vérification empirique : V_réel = U_static + traffic_servi_static (les deux somment à la même demande)
# Proxy : l'insatisfait statique doit toujours être ≥ l'insatisfait MILP
violations = df.filter(pl.col('u_milp_mo') > pl.col('u_static_mo') + 0.1)
print(f'Violations de logique (MILP > Statique) : {len(violations)} slots')
print('(Doit etre 0 — le MILP ne peut PAS faire pire que le statique sur ses propres cellules)')

# Vérification Oracle ≤ MILP (Oracle est une borne inférieure)
oracle_violations = df.filter(pl.col('u_oracle_mo') > pl.col('u_milp_mo') + 0.1)
print(f'\nViolations Oracle (Oracle > MILP) : {len(oracle_violations)} slots')
print('(Doit etre 0 — Oracle connait le futur donc il fait toujours mieux ou egal)')

## 4. Visualisations

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

FIG_DIR = ROOT / 'research/reports/sim48h'

fig, axes = plt.subplots(3, 1, figsize=(18, 22))
for ax, fig_name, title in zip(
    axes,
    ['fig1_congestion_profile.png', 'fig2_gain_per_slot.png', 'fig3_oracle_regret.png'],
    ['Fig 1 : Profil de congestion 48h',
     'Fig 2 : Gain MILP vs Statique par slot',
     'Fig 3 : Regret Oracle (zone orange = incertitude ML)']):
    img = mpimg.imread(str(FIG_DIR / fig_name))
    ax.imshow(img)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.show()

## 5. Comment reproduire ces résultats

```bash
# 1. Cloner le dépôt
git clone https://github.com/samya818/spatial-son-milp
cd spatial-son-milp

# 2. Installer les dépendances
pip install -r requirements.txt

# 3. Vérifier l'intégrité des données (SHA-256)
#    Le hash attendu est dans research/reports/sim48h/sim48h_metadata.json

# 4. Lancer la simulation (≈ 2-4h selon la machine)
python scripts/simulation_48h_rigorous.py

# 5. Résultats
#    research/reports/sim48h/sim48h_slots.csv     ← 96 lignes, une par slot
#    research/reports/sim48h/sim48h_summary.json  ← métriques agrégées
#    research/reports/sim48h/fig*.png             ← 3 figures
```

## 6. Questions anticipées du jury

**Q : Pourquoi MILP peut-il être ≥ Oracle sur certains slots ?**  
R : En théorie jamais. Numériquement, des écarts < 0.01 Mo peuvent apparaître à cause
des tolérances du solveur CBC. La section 3 ci-dessus vérifie automatiquement cela.

**Q : Pourquoi avoir choisi les jours 2013-11-12 et 2013-11-13 ?**  
R : Ce sont les 2 jours consécutifs avec le plus grand volume de trafic total du dataset
(88.35 To sur 48h). C'est le scénario le plus stressant pour le réseau — si WiseNet
fonctionne ici, il fonctionne à toutes les périodes.

**Q : Le XGBoost a-t-il vu ces jours pendant l'entraînement ?**  
R : Le split train/test du modèle est documenté dans `research/notebooks_v1_5/`.  
La variable `lag_*` garantit qu'aucune valeur future n'est utilisée comme feature.

**Q : Quelle est la valeur de l'Oracle dans la comparaison ?**  
R : L'Oracle quantifie le *regret d'incertitude* — la perte maximale due au fait qu'on
ne connaît pas le futur. Si l'efficacité ML est ≥ 95%, cela signifie que l'imperfection
de la prédiction ne coûte que 5% du gain potentiel.

**Q : La conservation de masse est-elle vraiment respectée ?**  
R : La section 3.2 + le test unitaire `tests/unit/test_camara.py` vérifient cela.
Voir la fonction `apply_decisions_on_real_traffic()` dans le script.